# Lesson 02 - Microsoft Agent Framework അന്വേഷണം

**Microsoft Agent Framework (MAF)** എആർ എജന്റുകൾ സൃഷ്ടിക്കുന്നതിനുള്ള ഏകീകൃത ഫ്രെയിംവർക്ക് ആണ്. ഇത് നാല് പ്രധാന ഘടകങ്ങളുമായി ശുദ്ധമായ, സമ്പ്രദായപരമായ ആർക്കിടെക്ചർ നൽകുന്നു:

- **ക്ലയന്റ്** – AI മോഡൽ എന്റ്പോയിന്റുമായി ബന്ധപ്പെടുകയും ആശയവിനിമയം നടത്തുകയും ചെയ്യുന്നു
- **എജന്റ്** – ക്ലയന്റിന് നിർദ്ദേശങ്ങളും ടൂൾ നിർവചനങ്ങളും ചേർത്ത് മറ.wrap
- **ടൂളുകൾ** – മോഡൽ വിളിക്കാവുന്ന ഇഷ്ടാനുസൃത ഫങ്ഷനുകളുമായി എജന്റ് കഴിവുകൾ വർദ്ധിപ്പിക്കുന്നു
- **സെഷൻ** – മൾടി-ടേൺ ഇടപെടലുകൾക്കായി സംഭാഷണ ചരിത്രം സൂക്ഷിക്കുന്നു

ഈ പാഠത്തിൽ, ഈ ആശയങ്ങൾ ഉപയോഗിച്ച് ഒരു **ട്രാവൽ ബുക്കിംഗ് ഏജന്റ്** നിർമ്മിക്കാം, അത് ലക്ഷ്യസ്ഥല ലഭ്യത പരിശോധിക്കുന്നു.


## സജ്ജീകരണം


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ഏജന്‍റ് ഫ്രെയിമ്വർക്കിന്റെ വാസ്തുവിദ്യയെ മനസ്സിലാക്കല്‍

Microsoft Agent Framework ഒരു അടിക്കെട്ടു വാസ്തുവിദ്യയെ പിന്തുടരുന്നു:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ക്ലയന്റ്** – ഒരു `AzureAIProjectAgentProvider` Azure OpenAI ഡിപ്പ്ലോയ്മെന്റുമായി കണക്ടുചെയ്യുന്നു. ഇത് പ്രാമാണീകരണം, അഭ്യർത്ഥന ഫോർമാറ്റിംഗ്, പ്രതികരണ പാഴ്സിംഗ് എന്നിവ കൈകാര്യം ചെയ്യുന്നു.
2. **ഏജന്‍റ്** – ക്ലയന്റിൽനിന്ന് `provider.create_agent()` വഴി സൃഷ്ടിക്കപ്പെട്ട ഏജന്‍റ് മോഡൽ ആക്‌സസ് സിസ്റ്റം പ്രോംപ്റ്റ് എന്നിവ ഒത്തുചേർന്ന് ഉപകരണങ്ങളടങ്ങിയതാണ്.
3. **ഉപകരണങ്ങള്‍** – `@tool` കൊണ്ട് അലങ്കരിച്ചിട്ടുള്ള Python ഫങ്ഷനുകളാണ്, ഏജന്‍റ് വഹിക്കാൻ കഴിയും പ്രവർത്തനങ്ങൾ നിർവഹിക്കാൻ അല്ലെങ്കിൽ ഡാറ്റ എടുക്കാൻ.
4. **സെഷൻ** – `agent.create_session()` വഴി സൃഷ്ടിക്കുന്ന `AgentSession` ഒബ്‌ജക്റ്റ്, സംഭാഷണ ചരിത്രം സൂക്ഷിച്ച് മൾട്ടി-ടേൺ ഡയലോഗ് സാധ്യമാക്കുന്നു, അതിൽ ഏജന്‍റ് മുൻപത്തെ പശ്ചാത്തലം ഓർക്കുന്നു.

ഓരോ പാളിയും ഘട്ടം ഘട്ടമായി നിർമ്മിക്കാം.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ഡെക്കറേറ്റർ ഉപയോഗിച്ച് ടൂളുകൾ ചേർക്കൽ

ടൂൾസുകൾ ഏജന്റുകൾക്ക് ടെക്സ്റ്റ് സൃഷ്‌ടിക്കുന്നതിലുപരി പ്രവർത്തനങ്ങൾ ചെയ്യാൻ അനുവദിക്കുന്നു. `@tool` ഡെക്കറേറ്റർ സാധാരണ പൈത്തൺ ഫംഗ്ഷൻ ഏജന്റ് വിളിക്കാവുന്ന ഒന്നായി മാറ്റുന്നു.

പ്രധാനം:
- ഓരോ പാരമീറ്ററിന്റെയും മോഡൽ മനസ്സിലാക്കാൻ `Annotated[type, "description"]` ഉപയോഗിക്കുക.
- ഡോക്സ്ട്രിംഗ് ടൂൾ വിവരണമായി മോഡലിന് കാണിക്കുന്നു.
- `approval_mode="never_require"` എന്നത് ഉപയോക്തൃ സ്ഥിരീകരണം ഇല്ലാതെ ടൂൾ സ്വയം പ്രവർത്തിക്കുമെന്ന് അർത്ഥം.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ടൂളുകളോടെ ഏജന്റ് സൃഷ്ടിക്കുന്നത്

ഇപ്പോൾ ക്ലയന്റും, നിർദ്ദേശങ്ങളും, ടൂളുകളും ചേർത്ത് ഒരു ഏജന്റ് ഉണ്ടാക്കുന്നു. `instructions` സിസ്റ്റം പ്രോംപ്റ്റ് ആയി പ്രവർത്തിക്കുന്നു — അവ ഏജന്റിന്റെ വ്യക്തിത്വവും പെരുമാറ്റവും നിർവചിക്കുന്നു.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## സെഷനുകളോടൊപ്പം ബഹു-മാറ്റം സംഭാഷണങ്ങൾ

ഒരു `AgentSession` (`agent.create_session()` വഴി സൃഷ്ടിച്ചത്) ഒരു സംഭാഷണത്തിലെ എല്ലാ സന്ദേശങ്ങളും ട്രാക്ക് ചെയ്യുന്നു. ഓരോ `agent.run()` കോളിലും ഒരേ സെഷൻ പാസ്സ് ചെയ്യുന്നതിലൂടെ, ഏജന്റ് മുഴുവൻ സംഭാഷണ ചരിത്രത്തെയും ആക്സസ് ചെയ്യുകയും മുൻസന്ദേശങ്ങളിലേക്ക് നിരീക്ഷണം നടത്തുകയും ചെയ്യാം.

ഞങ്ങൾ `tools=[check_destination_availability]` പാസ്സ് ചെയ്യുന്നതിലൂടെ ഏജന്റ് ഓരോ ടേണിലും നമ്മുടെ ലഭ്യത പരിശോധകത്തെ വിളിക്കാനാകും.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ പരambuപരambu Microsoft Agent Framework-ന്റെ നാല് സ്തംഭങ്ങൾ പരിശോധിച്ചു:

| ആശയം | നിങ്ങൾ പഠിച്ചത് |
|---------|------------------|
| **ക്ലയന്റ്** | `AzureAIProjectAgentProvider` ക്രെഡൻഷ്യൽ-പ്രാധാന്യമുള്ള auth ഉപയോഗിച്ച് Azure OpenAI-യുമായി കണക്ട് ചെയ്യുന്നു |
| **ഏജന്റ്** | `provider.create_agent()` മോഡൽ കണക്ഷനും നിർദ്ദേശങ്ങളും പേരും ചേർന്ന ഒരു പാക്കേജ് സൃഷ്ടിക്കുന്നു |
| **സാധനങ്ങൾ** | `@tool` ഡെക്കറേറ്റർ ഏജന്റിന് ഫോൺ ചെയ്യാനുള്ള Python ഫങ്ഷനുകൾ പ്രദർശിപ്പിക്കുന്നു |
| **സെഷൻ** | `agent.create_session()` ഒന്നിലധികം സംവാദം ഇടങ്ങളിലൂടെ സംഭാഷണ ചരിത്രം നിലനിർത്തുന്നു |

ഈ നിർമിത ഘടകങ്ങൾ ചേർന്ന് സ്വാഭാവികമായ സംഭാഷണങ്ങൾ നടത്താനും, ബാഹ്യ ഫങ്ഷനുകൾ വിളിക്കാനും, കോൺടെക്സ്റ്റ് നിലനിർത്താനുമായ ഏജენტები സൃഷ്ടിക്കുന്നു — പിന്നീട് പാഠങ്ങളിൽ കൂടുതൽ പരിണത ഏജന്റിക് പാറ്റേണുകൾക്കുള്ള അടിസ്ഥാനമാണ് ഇത്.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പദ്ധതി**:  
ഈ രേഖ [Co-op Translator](https://github.com/Azure/co-op-translator) എന്ന എഐ പരിഭാഷ സേവനം ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. നാം കൃത്യത പരിപാലിക്കാൻ ശ്രമിച്ചിരുന്നെങ്കിലും, സ്വയംപ്രവർത്തിത പരിഭാഷകളിൽ പിശകുകളോ അശുദ്ധികളോ ഉണ്ടായിരിക്കാം എന്ന് ദയവായി ശ്രദ്ധിക്കുക. പൊതുവായി, സ്വതന്ത്ര ഭാഷയിൽ ഉള്ള മേൽനോടുകൂടിയ ദസ്താവേസ് മാത്രമാണ് കൃത്യമായ ഉറവിടം എന്ന് പരിഗണിക്കപ്പെടുന്നത്. നിർണായക വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ നിർദ്ദേശിക്കുന്നു. ഈ പരിഭാഷയുടെ ഉപയോഗത്തിൽനിന്നുണ്ടാകാവുന്ന തെറ്റിദ്ധരിപ്പിക്കലുകൾക്കും തെറ്റ് വ്യാഖ്യാനങ്ങൾക്കുമായി ഞങ്ങൾ ഉത്തരവാദിത്തം ഏറ്റെടുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
